Fonte formato: https://www.emergentmind.com/topics/takum-format

In [ ]:
class Takum:
    def __init__(self, signal: int, direction: int, regime: int, mantissa: int, total_bits: int, exponent: int|None=None):
        self.signal = signal
        self.direction = direction
        self.regime = regime
        self.exponent = exponent
        self.mantissa = mantissa
        self.total_bits = total_bits
        
    @classmethod
    def from_string(cls, takum: str, total_bits: int):
        signal = int(takum[0], 2)
        direction = int(takum[1], 2)
        regime = int(takum[2:5], 2)
        exponent = None if regime == 0 else int(takum[5:(5+regime)], 2)
        mantissa = int(takum[(5+regime):], 2)
        return cls(signal, direction, regime, mantissa, total_bits, exponent)

In [ ]:
class UnpackedFormat:
    def __init__(self, signal: int, exponent: int, mantissa_grs: int, mantissa_length: int):
        self.signal = signal
        self.exponent = exponent
        self.mantissa_grs = mantissa_grs
        self.mantissa_length = mantissa_length

    @classmethod
    def from_takum(cls, takum: Takum):
        signal = takum.signal
        exponent = cls.get_unbiased_exponent(takum)
        mantissa_grs, mantissa_length = cls.get_full_mantissa(takum)
        return cls(signal, exponent, mantissa_grs, mantissa_length)

    @staticmethod
    def get_unbiased_exponent(takum: Takum):
        e = takum.exponent
        r = takum.regime
        d = takum.direction

        base_scale = 0 if r == 0 else (1 << (r + 1)) - 2

        if d == 0:
            base_scale = -base_scale - 1
            unbiased_exponent = base_scale - e if e is not None else base_scale
        else:
            unbiased_exponent = base_scale + e if e is not None else base_scale

        return unbiased_exponent

    @staticmethod
    def get_full_mantissa(takum: Takum) -> tuple[int, int]:
        mantissa_length = takum.total_bits - (5 + takum.regime)
        
        if takum.mantissa == 0:
            return 0, mantissa_length

        true_significand = (1 << mantissa_length) | takum.mantissa

        # adds bits G, R and S
        grs_added = true_significand << 3
        return grs_added, mantissa_length

    def copy(self):
        return UnpackedFormat(self.signal, self.exponent, self.mantissa_grs, self.mantissa_length)

In [ ]:
def add(a: UnpackedFormat, b: UnpackedFormat, total_bits: int):
    is_a_bigger = a.exponent > b.exponent
    bigger = a.copy() if is_a_bigger else b.copy()
    smaller = b.copy() if is_a_bigger else a.copy()
    
    delta = bigger.exponent - smaller.exponent

    if(bigger.exponent > smaller.exponent):
        mask = (1 << delta) - 1
        dropped_bits = smaller.mantissa_grs & mask
        
        smaller.mantissa_grs >>= delta
        
        if dropped_bits > 0:
            smaller.mantissa_grs |= 1

    if(a.signal == b.signal):
        result_mantissa = a.mantissa_grs + b.mantissa_grs
        result_signal = a.signal
    elif(a.mantissa_grs > b.mantissa_grs):
        result_mantissa = a.mantissa_grs - b.mantissa_grs
        result_signal = a.signal
    elif(b.mantissa_grs > a.mantissa_grs):
        result_mantissa = b.mantissa_grs - a.mantissa_grs
        result_signal = b.signal
    else:
        result_mantissa = 0
        result_signal = 0

    if result_mantissa == 0:
        return Takum(0, 0, 0, 0, total_bits, 0)

    hidden_bit_position = a.mantissa_length + 3
    current_length = result_mantissa.bit_length()
    shift_delta = current_length - 1 - hidden_bit_position

    normalized_mantissa = result_mantissa
    normalized_exponent = bigger.exponent

    if shift_delta > 0:
        bits_to_drop_mask = (1 << shift_delta) - 1 
        dropped_bits = result_mantissa & bits_to_drop_mask

        normalized_mantissa = result_mantissa >> shift_delta

        if dropped_bits > 0:
            normalized_mantissa |= 1

        normalized_exponent += shift_delta

    elif shift_delta < 0:
        shift_amount = abs(shift_delta)

        normalized_mantissa = result_mantissa << shift_amount
        normalized_exponent -= shift_amount

    grs_bits = normalized_mantissa & 0b111 
    final_mantissa = normalized_mantissa >> 3
    
    if grs_bits >= 0b100:
        final_mantissa += 1
        if final_mantissa & (final_mantissa - 1) == 0:
            final_mantissa >>= 1
            normalized_exponent += 1
        
    result_direction = 1 if normalized_exponent >= 0 else 0
    result_regime = 0
    result_exponent = 0

    abs_exp = abs(normalized_exponent)
    if abs_exp > 0:
        for r in range(1, 8):
            base_scale = (1 << (r + 1)) - 2
            max_exp_value = (1 << r) - 1
            max_regime_capacity = base_scale + max_exp_value
            
            if abs_exp <= max_regime_capacity:
                result_regime = r
                
                result_exponent = abs(abs_exp - base_scale)
                break
    
    final_length = final_mantissa.bit_length()
    mask_remove_hidden = (1 << (final_length - 1)) - 1
    final_mantissa &= mask_remove_hidden

    return Takum(result_signal, result_direction, result_regime, final_mantissa, total_bits, result_exponent)

In [ ]:
def sub(a: UnpackedFormat, b: UnpackedFormat, total_bits: int):
    simetric_b = b.copy()
    simetric_b.signal ^= 1
    return add(a, simetric_b, total_bits)

In [ ]:
def mul(a: UnpackedFormat, b: UnpackedFormat, total_bits: int):
    if a.mantissa_grs == 0 or b.mantissa_grs == 0:
        return Takum(0, 0, 0, 0, total_bits, 0)

    result_signal = a.signal ^ b.signal
    
    raw_mantissa = a.mantissa_grs * b.mantissa_grs
    current_length = raw_mantissa.bit_length()
    
    normalized_exponent = (a.exponent + b.exponent + current_length - a.mantissa_length - b.mantissa_length - 7)

    target_hidden_bit = max(a.mantissa_length, b.mantissa_length) + 3
    shift_delta = (current_length - 1) - target_hidden_bit
    
    normalized_mantissa = raw_mantissa

    if shift_delta > 0:
        mask = (1 << shift_delta) - 1
        dropped_bits = raw_mantissa & mask
        
        normalized_mantissa = raw_mantissa >> shift_delta
        
        if dropped_bits > 0:
            normalized_mantissa |= 1
            
    elif shift_delta < 0:
        normalized_mantissa = raw_mantissa << abs(shift_delta)

    grs_bits = normalized_mantissa & 0b111 
    final_mantissa = normalized_mantissa >> 3
    
    if grs_bits >= 0b100:
        final_mantissa += 1
        if final_mantissa & (final_mantissa - 1) == 0:
            final_mantissa >>= 1
            normalized_exponent += 1
            
    result_direction = 1 if normalized_exponent >= 0 else 0
    result_regime = 0
    result_exponent = 0

    abs_exp = abs(normalized_exponent)
    if abs_exp > 0:
        for r in range(1, 8):
            base_scale = (1 << (r + 1)) - 2
            max_exp_value = (1 << r) - 1
            max_regime_capacity = base_scale + max_exp_value
            
            if abs_exp <= max_regime_capacity:
                result_regime = r
                result_exponent = abs(abs_exp - base_scale)
                break
    
    final_length = final_mantissa.bit_length()
    shift_amount = max(0, final_length - 1)
    mask_remove_hidden = (1 << shift_amount) - 1
    
    final_mantissa &= mask_remove_hidden

    return Takum(result_signal, result_direction, result_regime, final_mantissa, total_bits, result_exponent)

In [ ]:
def vtsetup(a: list[Takum], b: list[Takum]):
    if not a:
        return [], [], 0

    unpacked_a: list[UnpackedFormat] = []
    unpacked_b: list[UnpackedFormat] = []
    
    for i in range(len(a)):
        unpacked_a.append(UnpackedFormat.from_takum(a[i]))
        unpacked_b.append(UnpackedFormat.from_takum(b[i]))
    
    total_bits = a[0].total_bits

    return unpacked_a, unpacked_b, total_bits

In [ ]:
def vtadd(a: list[Takum], b: list[Takum]):
    if len(a) != len(b):
        raise ValueError('The arrays to be summed must have the same length.')

    unpacked_a, unpacked_b, total_bits = vtsetup(a, b)
    result: list[Takum] = []
    for i in range(len(a)):
        result.append(add(unpacked_a[i], unpacked_b[i], total_bits))

    return result

In [ ]:
def vtsub(a: list[Takum], b: list[Takum]):
    if len(a) != len(b):
        raise ValueError('The arrays to be subtracted must have the same length.')

    unpacked_a, unpacked_b, total_bits = vtsetup(a, b)
    result: list[Takum] = []
    for i in range(len(a)):
        result.append(sub(unpacked_a[i], unpacked_b[i], total_bits))

    return result

In [ ]:
def vtmul(a: list[Takum], b: list[Takum]):
    if len(a) != len(b):
        raise ValueError('The arrays to be multiplied must have the same length.')

    unpacked_a, unpacked_b, total_bits = vtsetup(a, b)
    result: list[Takum] = []
    for i in range(len(a)):
        result.append(mul(unpacked_a[i], unpacked_b[i], total_bits))

    return result

In [ ]:
def vtdot(a: list[Takum], b: list[Takum]):
    if len(a) != len(b):
        raise ValueError('The arrays to be MAC`d must have the same length.')

    unpacked_a, unpacked_b, total_bits = vtsetup(a, b)
    acc = Takum(0, 0, 0, 0, total_bits, 0)

    for i in range(len(a)):
        product = mul(unpacked_a[i], unpacked_b[i], total_bits)

        unpacked_acc = UnpackedFormat.from_takum(acc)
        unpacked_product = UnpackedFormat.from_takum(product)

        acc = add(unpacked_acc, unpacked_product, total_bits)

    return acc